# **4. REINFORCE para _cartpole_**

En este cuaderno, vamos a resolver [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/), el mismo problema de [02_cartpole-dqn.ipynb](02_cartpole-dqn.ipynb), con **REINFORCE**, un algoritmo de gradiente de política. En lugar de aprender el valor de cada acción, como DQN, REINFORCE aprende directamente la probabilidad de elegir cada acción en cada estado.

Basado en:
> Udacity, Deep Reinforcement Learning Course. Disponible en [GitHub](https://github.com/udacity/deep-reinforcement-learning/tree/master/reinforce).

## **4.0. Preparación**

Comenzamos importando las bibliotecas necesarias, así como otras utilidades que emplearemos más adelante:

In [ ]:
import os
import gymnasium as gym
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
from collections import deque

import io
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, Image as IPImage
from PIL import Image

def show_state(env):
    """
    Muestra el estado actual del entorno.
    """
    img = env.render()
    clear_output(wait=True)
    plt.imshow(img)
    plt.axis("off")
    plt.show()

def show_episode(env, act):
    """
    Ejecuta un episodio completo y lo muestra como una animación.

    Parámetros
    ==========
        env: entorno creado con render_mode="rgb_array"
        act (callable): función que devuelve la acción a realizar en un estado
    """
    state, info = env.reset()
    frames = [Image.fromarray(env.render())]
    episode_reward = 0
    terminated = truncated = False

    # Bucle de interacción agente-entorno (1 episodio)
    while not (terminated or truncated):
        action = act(state)
        state, reward, terminated, truncated, info = env.step(action)
        frames.append(Image.fromarray(env.render()))
        episode_reward += reward

    # Guardar las imágenes como un GIF animado y mostrarlo
    gif = io.BytesIO()
    frames[0].save(gif, format="gif", save_all=True, append_images=frames[1:],
                   duration=1000 // env.metadata["render_fps"], loop=0)
    display(IPImage(data=gif.getvalue(), format="gif"))
    print("Recompensa del episodio: ", episode_reward)

def plot_rewards(episode_rewards, title, solved_reward=None, window=100):
    """
    Representa la recompensa de cada episodio de entrenamiento y la media móvil de los últimos `window`.

    Parámetros
    ==========
        episode_rewards (list): recompensa de cada episodio de entrenamiento
        title (str): título de la gráfica
        solved_reward (float): recompensa media para considerar resuelto el problema (opcional)
        window (int): número de episodios de la media móvil
    """
    episode_rewards = np.asarray(episode_rewards)
    avg = [np.mean(episode_rewards[max(0, i-window+1):i+1]) for i in range(len(episode_rewards))]
    plt.plot(np.arange(len(episode_rewards)), episode_rewards, alpha=0.4, label="Recompensa")
    plt.plot(np.arange(len(episode_rewards)), avg, label=f"Recompensa media (últimos {window})")
    if solved_reward is not None:
        plt.axhline(solved_reward, color="gray", linestyle="--", label="Resuelto")
    plt.ylabel("Recompensa")
    plt.xlabel("Episodio nº")
    plt.title(title)
    plt.legend()
    plt.show()

## **4.1. Entorno**

Creamos el entorno [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/). El estado es un vector de $4$ valores continuos (posición y velocidad del carro, ángulo y velocidad angular del péndulo) y hay $2$ acciones posibles: `0` (empujar el carro a la izquierda) y `1` (empujarlo a la derecha).

In [ ]:
env = gym.make("CartPole-v1", render_mode="rgb_array")

print(env.observation_space)
print(env.action_space)

Una vez creado el entorno, podemos visualizar el **estado inicial** de la simulación:

In [ ]:
# Inicializar el entorno
env.reset()

# Mostrar estado inicial del entorno
show_state(env)

La función `show_episode()` muestra un episodio como una animación, junto con la **recompensa del episodio**, es decir, la suma de las recompensas obtenidas en cada paso. Veamos, por ejemplo, un episodio de un agente que elige acciones al azar:

In [ ]:
show_episode(env, lambda state: env.action_space.sample())

## **4.2. Algoritmo REINFORCE**

A continuación, proporcionamos una implementación genérica de REINFORCE. Seguimos la misma estructura que en [02_cartpole-dqn.ipynb](02_cartpole-dqn.ipynb) pero, como el código es mucho más corto, lo incluimos en el cuaderno en lugar de en ficheros externos:

*   `Policy`: Implementación de la red neuronal del agente (la red de política), equivalente a [model.py](model.py) en DQN.
*   `Agent`: Implementación de las capacidades del agente (`act`, `step`, `learn`), equivalente a [dqn_agent.py](dqn_agent.py) en DQN.

El entorno [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/) se considera resuelto cuando el agente obtiene una recompensa media de al menos $475$ en 100 episodios consecutivos. Los episodios se truncan a los $500$ pasos, que es por tanto la recompensa máxima de un episodio.

### **4.2.1. Red de política**

A diferencia de la red Q de DQN, que estima el valor de cada acción, la red de política devuelve directamente la probabilidad de seleccionar cada acción.

In [ ]:
class Policy(nn.Module):
    """Red de política: asigna a cada estado las probabilidades de las acciones."""

    def __init__(self, state_size, action_size, seed, fc1_units=16):
        """Inicializa los parámetros y construye el modelo.
        Parámetros
        ==========
            state_size (int): dimensión de cada estado
            action_size (int): dimensión de cada acción
            seed (int): semilla aleatoria
            fc1_units (int): número de nodos de la capa oculta
        """
        super(Policy, self).__init__()
        torch.manual_seed(seed)
        self.fc1 = nn.Linear(state_size, fc1_units)
        self.fc2 = nn.Linear(fc1_units, action_size)

    def forward(self, state):
        """Construye una red que asigna estado -> probabilidades de las acciones."""
        x = F.relu(self.fc1(state))
        return F.softmax(self.fc2(x), dim=1)

### **4.2.2. Agente**

La principal diferencia con el agente DQN es _cuándo_ aprende. El agente DQN aprende en cada paso a partir de un minilote de experiencias pasadas almacenadas en la memoria de experiencias (_replay buffer_). El agente REINFORCE solo almacena el logaritmo de la probabilidad de cada acción y la recompensa obtenida (`step`), y aprende al final de cada episodio (`learn`), cuando ya puede calcularse el retorno descontado desde cada paso en adelante, $G_t$.

Para reducir la varianza de la estimación del gradiente, cada acción se pondera por el $G_t$ (normalizado) en lugar de por el retorno total del episodio.

In [ ]:
GAMMA = 0.99            # Factor de descuento (gamma)
LR = 1e-2               # Tasa de aprendizaje (eta)

class Agent():
    """Interactúa con el entorno y aprende de él."""

    def __init__(self, state_size, action_size, seed):
        """Inicializa un objeto Agent.

        Parámetros
        ==========
            state_size (int): dimensión de cada estado
            action_size (int): dimensión de cada acción
            seed (int): semilla aleatoria
        """
        self.state_size = state_size
        self.action_size = action_size
        random.seed(seed)

        # Red de política
        self.policy = Policy(state_size, action_size, seed)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=LR)

        # Trayectoria del episodio actual
        self.saved_log_probs = []  # Logaritmos de las probabilidades de las acciones tomadas
        self.rewards = []          # Recompensas obtenidas

    def act(self, state):
        """Muestrea una acción para el estado dado según la política actual.

        Parámetros
        ==========
            state (array_like): estado actual
        Devuelve
        ========
            action (int): acción muestreada
            log_prob (torch.Tensor): logaritmo de la probabilidad de la acción (necesario para aprender)
        """
        state = torch.from_numpy(state).float().unsqueeze(0)
        probs = self.policy(state)
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action)

    def step(self, log_prob, reward):
        """Almacena el logaritmo de la probabilidad de la acción tomada (At) y la recompensa obtenida (Rt+1).

        Parámetros
        ==========
            log_prob (torch.Tensor): logaritmo de la probabilidad de la acción tomada
            reward (float): recompensa obtenida
        """
        self.saved_log_probs.append(log_prob)
        self.rewards.append(reward)

    def learn(self, gamma=GAMMA):
        """Actualiza los parámetros de la política con la trayectoria del episodio que acaba de terminar.

        Parámetros
        ==========
            gamma (float): factor de descuento
        """
        # Calcular el retorno descontado desde cada paso en adelante: G_t = R_t+1 + gamma * G_t+1
        returns = deque()
        G = 0
        for reward in reversed(self.rewards):
            G = reward + gamma * G
            returns.appendleft(G)

        # Normalizar los retornos (reduce la varianza de la estimación del gradiente)
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + 1e-9)

        # Calcular la pérdida y actualizar
        policy_loss = []
        for log_prob, G in zip(self.saved_log_probs, returns):
            policy_loss.append(-log_prob * G)
        policy_loss = torch.cat(policy_loss).sum()

        self.optimizer.zero_grad()
        policy_loss.backward()
        self.optimizer.step()

        # Empezar una nueva trayectoria
        self.saved_log_probs = []
        self.rewards = []

### **4.2.3. Bucle de entrenamiento**

El bucle de entrenamiento es casi igual que en DQN. La única diferencia es la llamada a `agent.learn()` al final de cada episodio.

In [ ]:
SOLVED_REWARD = 475.0   # Recompensa media (últimos 100 episodios) para considerar resuelto CartPole-v1

def reinforce(agent, n_episodes=1000, print_every=100):
    """REINFORCE.

    Parámetros
    ==========
        agent (Agent): agente a entrenar
        n_episodes (int): número máximo de episodios
        print_every (int): mostrar la recompensa media cada print_every episodios
    """
    episode_rewards = []                # Recompensas de todos los episodios
    rewards_window = deque(maxlen=100)  # Recompensas de los últimos 100 episodios

    for i_episode in range(1, n_episodes+1):
        state, info = env.reset()
        episode_reward = 0
        terminated = truncated = False

        # Bucle de interacción agente-entorno (1 episodio)
        while not (terminated or truncated):
            # Elegir la acción At con la red de política
            action, log_prob = agent.act(state)

            # Aplicar At y obtener Rt+1, St+1
            next_state, reward, terminated, truncated, info = env.step(action)

            # Almacenar el logaritmo de la probabilidad de At y Rt+1
            agent.step(log_prob, reward)

            # Avanzar al siguiente estado
            state = next_state
            episode_reward += reward

        # Entrenar y actualizar con la trayectoria completa
        agent.learn()

        rewards_window.append(episode_reward)   # Guardar la recompensa del episodio (ventana de 100)
        episode_rewards.append(episode_reward)  # Guardar la recompensa del episodio (todos los episodios)

        # Mostrar información del entrenamiento
        if i_episode % print_every == 0:
            print(f"Episodio {i_episode}\tRecompensa media (últimos 100): {np.mean(rewards_window):.2f}")
        if len(rewards_window) == 100 and np.mean(rewards_window) >= SOLVED_REWARD:
            print(f"\n¡Problema resuelto en {i_episode-100} episodios!\tRecompensa media (últimos 100): {np.mean(rewards_window):.2f}")
            break
    else:
        print(f"\nProblema no resuelto en {n_episodes} episodios.\tRecompensa media (últimos 100): {np.mean(rewards_window):.2f}")

    os.makedirs("agentes", exist_ok=True)
    torch.save(agent.policy.state_dict(), "agentes/cartpole-reinforce.pth") # Guardar los pesos del agente entrenado

    return episode_rewards

agent = Agent(state_size=4, action_size=2, seed=0)
episode_rewards = reinforce(agent, n_episodes=5000)

plot_rewards(episode_rewards, "REINFORCE", SOLVED_REWARD)

## **4.3. Visualizar el agente entrenado**

Cargamos los pesos en el agente y lo ejecutamos durante un episodio, muestreando las acciones de la política aprendida:

In [ ]:
# Cargar los pesos desde `agentes/cartpole-reinforce.pth`
agent_load = Agent(state_size=4, action_size=2, seed=0)

agent_load.policy.load_state_dict(torch.load("agentes/cartpole-reinforce.pth", weights_only=True))

show_episode(env, lambda state: agent_load.act(state)[0])   # act() devuelve la acción y su log-probabilidad

¡Nuestro agente ha aprendido de nuevo a mantener el péndulo en equilibrio, esta vez aprendiendo directamente la política!

En este cuaderno, hemos implementado REINFORCE, un algoritmo de gradiente de política, con la misma estructura de agente que DQN: la principal diferencia es que aprende al final de cada episodio, a partir de los retornos obtenidos.

En los siguientes cuadernos, resolveremos los mismos problemas con las implementaciones de la biblioteca Stable-Baselines3.